In [ ]:
import os
from google.colab import userdata, drive
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")

!git clone -q https://github.com/kaist-dmlab/QuDAR.git /content/QuDAR
!pip install -q -e /content/QuDAR
!unzip -q -o /content/qudar-patches.zip -d /content
!python /content/qudar-patches/apply_patch.py /content/QuDAR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/QuDAR' already exists and is not an empty directory.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for qudar (pyproject.toml) ... done
patched /content/QuDAR/src/qudar/retrievers.py


In [ ]:
from beir import util
for name in ["fiqa", "scidocs"]:
    util.download_and_unzip(f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip", "/content/data")
DATA = "/content/data"

for ds in ["fiqa", "scidocs"]:
    !python -m qudar.build_index --dataset-path {DATA}/{ds} --model-name facebook/contriever \
        --output-dir /content/indexes/{ds}/contriever --dim 768 --batch-size 128

/usr/local/lib/python3.13/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


/content/data/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

/content/data/scidocs.zip:   0%|          | 0.00/136M [00:00<?, ?iB/s]

100% 57638/57638 [00:00<00:00, 98255.75it/s]
config.json: 100% 619/619 [00:00<00:00, 2.27MB/s]
tokenizer_config.json: 100% 321/321 [00:00<00:00, 1.30MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 4.61MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 12.0MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 514kB/s]

pytorch_model.bin: downloading bytes:  46% 202M/438M [00:01<00:00, 243MB/s, 15.8MB/s  ]
pytorch_model.bin: downloading bytes:  82% 359M/438M [00:02<00:00, 182MB/s, 31.1MB/s  ]
pytorch_model.bin: downloading bytes:  87% 381M/438M [00:02<00:00, 189MB/s, 32.0MB/s  ]
pytorch_model.bin: downloading bytes:  93% 407M/438M [00:02<00:00, 166MB/s, 34.7MB/s  ]
pytorch_model.bin: downloading bytes: 100% 413M/413M [00:02<00:00, 148MB/s, 35.8MB/s  ]
pytorch_model.bin: reconstructing file: 100% 438M/438M [00:02<00:00, 156MB/s, 39.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 16945.48it/s]

model.safetensors: downloading bytes:   0% 0.00/438M [00:00<?, ?B/s]

Dense encoding:   0

In [ ]:
import sys; sys.path.insert(0, "/content/QuDAR/src")
from qudar.data import load_beir_dataset
from qudar.retrievers import build_sparse_retriever, DenseRetriever
from qudar.evaluation import evaluate_run

def to_run(qids, results):
    return {q: {d: float(s) for d, s in zip(docs, scores)} for q, (docs, scores) in zip(qids, results)}

for ds in ["fiqa", "scidocs"]:
    b = load_beir_dataset(f"{DATA}/{ds}", "test")
    qids = list(b.queries); qtexts = [b.queries[q] for q in qids]
    bm25 = build_sparse_retriever("bm25", b.corpus, 1000)
    print(ds, "OS", evaluate_run(b.qrels, to_run(qids, bm25.search(qtexts, 1000)), 10)[0])
    dense = DenseRetriever("facebook/contriever", f"/content/indexes/{ds}/contriever", 768)
    print(ds, "OD", evaluate_run(b.qrels, to_run(qids, dense.search(dense.encode_queries(qtexts), 1000)), 10)[0])

  0%|          | 0/57638 [00:00<?, ?it/s]

BM25 retrieval: 100%|██████████| 648/648 [00:00<00:00, 945.55it/s]


fiqa OS {'Recall@10': 0.2156, 'nDCG@10': 0.1678, 'MAP@10': 0.1238, 'Precision@1': 0.1481, 'MRR': 0.218, 'Hit@10': 0.3549}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense retrieval: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]


fiqa OD {'Recall@10': 0.1492, 'nDCG@10': 0.1188, 'MAP@10': 0.0839, 'Precision@1': 0.1096, 'MRR': 0.1702, 'Hit@10': 0.2855}


... (2 outputs left out)

scidocs OS {'Recall@10': 0.1412, 'nDCG@10': 0.139, 'MAP@10': 0.0806, 'Precision@1': 0.175, 'MRR': 0.2661, 'Hit@10': 0.447}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense retrieval: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


scidocs OD {'Recall@10': 0.095, 'nDCG@10': 0.0889, 'MAP@10': 0.0463, 'Precision@1': 0.102, 'MRR': 0.1823, 'Hit@10': 0.349}


In [ ]:
!mkdir -p /content/drive/MyDrive/qudar && cp -r /content/indexes /content/drive/MyDrive/qudar/

In [ ]:
!pip install -q bm25s
import sys; sys.path.insert(0, "/content/QuDAR/src")
import bm25s, Stemmer
from qudar.data import load_beir_dataset
from qudar.evaluation import evaluate_run
stemmer = Stemmer.Stemmer("english")

def bm25s_run(b, method, stopwords):
    ids = list(b.corpus)
    texts = [(" ".join(p for p in [b.corpus[i].get("title"), b.corpus[i].get("text")] if p)).strip() for i in ids]
    r = bm25s.BM25(method=method)
    r.index(bm25s.tokenize(texts, stopwords=stopwords, stemmer=stemmer, show_progress=False), show_progress=False)
    qids = list(b.queries)
    qtok = bm25s.tokenize([b.queries[q] for q in qids], stopwords=stopwords, stemmer=stemmer, return_ids=False, show_progress=False)
    docs, scores = r.retrieve(qtok, k=1000, show_progress=False)
    return {q: {ids[d]: float(s) for d, s in zip(docs[i], scores[i])} for i, q in enumerate(qids)}

for ds in ["fiqa", "scidocs"]:
    b = load_beir_dataset(f"{DATA}/{ds}", "test")
    for name, kw in [("lucene + stopwords", dict(method="lucene", stopwords="en")),
                     ("robertson + stopwords", dict(method="robertson", stopwords="en")),
                     ("lucene, no stopwords", dict(method="lucene", stopwords=None))]:
        print(ds, "|", name, "|", evaluate_run(b.qrels, bm25s_run(b, **kw), 10)[0])

  0%|          | 0/57638 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


fiqa | lucene + stopwords | {'Recall@10': 0.3178, 'nDCG@10': 0.2514, 'MAP@10': 0.19, 'Precision@1': 0.2299, 'MRR': 0.319, 'Hit@10': 0.4907}


DEBUG:bm25s:Building index from IDs objects


fiqa | robertson + stopwords | {'Recall@10': 0.3192, 'nDCG@10': 0.2528, 'MAP@10': 0.1912, 'Precision@1': 0.2346, 'MRR': 0.3218, 'Hit@10': 0.4923}


DEBUG:bm25s:Building index from IDs objects


... (4 outputs left out)

DEBUG:bm25s:Building index from IDs objects


scidocs | robertson + stopwords | {'Recall@10': 0.1659, 'nDCG@10': 0.1581, 'MAP@10': 0.0924, 'Precision@1': 0.183, 'MRR': 0.2871, 'Hit@10': 0.497}


DEBUG:bm25s:Building index from IDs objects


scidocs | lucene, no stopwords | {'Recall@10': 0.1637, 'nDCG@10': 0.1561, 'MAP@10': 0.0913, 'Precision@1': 0.178, 'MRR': 0.2835, 'Hit@10': 0.491}


In [ ]:
import gc
import torch
from qudar.retrievers import DenseEncoder
from qudar.index import DenseIndexer

# Free up unused CUDA memory
gc.collect()
torch.cuda.empty_cache()

b = load_beir_dataset(f"{DATA}/fiqa", "test")
enc = DenseEncoder("facebook/contriever", query_prefix="", passage_prefix="")
ids = list(b.corpus)

# Lowered batch_size from 128 to 32 to prevent CUDA Out of Memory errors
idx = DenseIndexer(768); idx.index_data(ids, enc.encode_corpus([b.corpus[i] for i in ids], batch_size=32))
qids = list(b.queries)
res = idx.search(enc.encode_queries([b.queries[q] for q in qids]), 1000)
print("fiqa OD no-prefix", evaluate_run(b.qrels, {q: {d: float(s) for d, s in zip(*r)} for q, r in zip(qids, res)}, 10)[0])

  0%|          | 0/57638 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


fiqa OD no-prefix {'Recall@10': 0.1525, 'nDCG@10': 0.1241, 'MAP@10': 0.088, 'Precision@1': 0.1235, 'MRR': 0.1829, 'Hit@10': 0.2917}
